In [1]:
from core.model_providers.openai_llm import OpenAILLM
from core import KAGConfig
from core.storage.graph_store import GraphStore
from core.utils.neo4j_utils import Neo4jUtils

config = KAGConfig.from_yaml("configs/config_openai.yaml")
llm = OpenAILLM(config)
doc_type = config.knowledge_graph_builder.doc_type
graph_store = GraphStore(config)
neo4j_utils = Neo4jUtils(graph_store.driver, doc_type=doc_type)

/vepfs-mlp2/c20250513/241404044/users/roytian/anaconda3/envs/screenplay/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
document_list = neo4j_utils.fetch_all_nodes(node_types=["Scene"])

In [3]:
document_list[0]

{'labels': ['Entity', 'Scene'],
 'id': 'doc_0',
 'name': '1 INT.  COMMUNITY HALL.   DAY.',
 'description': 'John embraces a 600-mile journey as a positive change, feeling empowered and grateful to his fellowship for supporting his path to freedom, not escape.',
 'properties': '{"scene_name": "1 INT.  COMMUNITY HALL.   DAY.", "source_doc_id": "doc_0", "document_type": "screenplay", "metadata": {"doc_title": "1 INT.  COMMUNITY HALL.   DAY.", "title": "1 INT.  COMMUNITY HALL.   DAY.", "subtitle": "", "order": 0, "version": "default", "scene_id": "1", "scene_category": "INT", "lighting": "Day", "space": "real world", "region": null, "main_location": "Community Hall", "sub_location": null, "summary": "John embraces a 600-mile journey as a positive change, feeling empowered and grateful to his fellowship for supporting his path to freedom, not escape.", "source_doc_id": "doc_0", "source_title": "1 INT.  COMMUNITY HALL.   DAY."}}',
 'source_documents': ['scene_1_part_1', 'scene_1_part_2']}

In [4]:
related_entities = neo4j_utils.search_related_entities(source_id="doc_0", entity_types=["Event", "Occasion"])

In [5]:
related_entities

[Entity(id='ent_b4a39cf252a9', name='Narrator shares personal revelation with fellowship (scene_1)', type=['Event'], aliases=[], properties={}, description='The narrator stops to express gratitude and announce a significant life change during a fellowship meeting, affirming a sense of freedom and purpose.', scope='local', source_documents=['scene_1_part_2']),
 Entity(id='ent_422301c9f29b', name='Fellowship Meeting (scene_1)', type=['Occasion'], aliases=[], properties={}, description='A gathering of individuals in a support fellowship where the narrator shares a personal milestone and expresses gratitude before embarking on a journey.', scope='local', source_documents=['scene_1_part_2']),
 Entity(id='ent_7a1376a86fef', name='Narrator begins 600-mile journey (scene_1)', type=['Event'], aliases=[], properties={}, description='The narrator starts walking a long-distance journey of 600 miles, marking a personal turning point and commitment to change.', scope='local', source_documents=['scen

In [6]:
def get_entity_info(entities):
    entity_info = ""
    for entity in entities:
        entity_info += f"- id: {entity.id}, "
        entity_info += f"  name: {entity.name}, "
        entity_info += f"  type: {entity.type}, "
        entity_info += f"  description: {entity.description}\n"
    return entity_info

In [7]:
from core.builder.manager.narrative_analysis_manager import NarrativeManager

narrative_manager = NarrativeManager(config, llm)

In [8]:
import json
from tqdm import tqdm

with open("data/knowledge_graph/doc2chunks.json", "r") as f:
    doc2chunks = json.load(f)

reached_entities = []
document_list = neo4j_utils.fetch_all_nodes(node_types=["Scene"])
source_documents = document_list[0]["source_documents"]
all_entities = neo4j_utils.search_related_entities(source_id="doc_0", entity_types=["Event", "Occasion"])

for i, doc_id in  tqdm(enumerate(source_documents), desc="Processing Episode Extraction", total=len(source_documents)):
    if i == 0:
        related_entities = [ent for ent in all_entities if ent.source_documents[0]==doc_id]
        reached_entities.extend(related_entities)
        entity_info = get_entity_info(reached_entities)
        text = "\n".join([chunk.get("content") for chunk in doc2chunks[doc_id]["chunks"]])
        output = narrative_manager.extract_episodes(text=text, entities=entity_info, goal="extract")
        existing_episodes = json.loads(output)["episodes"]

    else:
        related_entities = [ent for ent in all_entities if ent.source_documents[0]==doc_id]
        reached_entities.extend(related_entities)
        entity_info = get_entity_info(reached_entities)
        text = "\n".join([chunk.get("content") for chunk in doc2chunks[doc_id]["chunks"]])
        existing_episodes_str = json.dumps(existing_episodes, ensure_ascii=False, indent=2)
        output = narrative_manager.extract_episodes(text=text, entities=entity_info, 
                                                goal="update", existing_episodes=existing_episodes_str)
        existing_episodes = json.loads(output)["episodes"]


Processing Episode Extraction:   0%|          | 0/2 [00:00<?, ?it/s]

Processing Episode Extraction: 100%|██████████| 2/2 [00:13<00:00,  6.99s/it]


In [9]:
source_documents

['scene_1_part_1', 'scene_1_part_2']

In [10]:
neo4j_utils.search_related_entities("ent_dede62517252", entity_types=["Character"])

[Entity(id='ent_2d0db79cb920', name='Benny', type=['Character'], aliases=[], properties={'gender': 'male', 'identity': 'recovering alcoholic; member of Alcoholics Anonymous', 'affiliation': 'Alcoholics Anonymous', 'role_in_community': 'peer support through shared experience'}, description="Benny is a man committed to his recovery from alcohol addiction through active participation in Alcoholics Anonymous. He has a history of struggling with alcohol, including periods of secretive drinking, and now shares his personal journey at AA meetings. His involvement reflects a sustained effort to maintain sobriety and support others facing similar challenges. Benny's role in the community centers on offering experience and encouragement to fellow members.", scope='global', source_documents=['scene_1_part_1'])]

In [11]:
def get_event_context(event_id):
    characters = neo4j_utils.search_related_entities(event_id, entity_types=["Character"])
    locations = neo4j_utils.search_related_entities(event_id, entity_types=["Location"])
    times = neo4j_utils.search_related_entities(event_id, entity_types=["TimePoint"])

    context = {
        "characters": [ent.name for ent in characters],
        "locations": [ent.name for ent in locations],
        "times": [ent.name for ent in times]
    }
    return context

In [12]:
import hashlib
existing_episodes_info = []
for episide in existing_episodes:
    name = episide["name"]
    description = episide["description"]
    eid = hashlib.md5(
            f"{name}{description}".encode("utf-8")
        ).hexdigest()[:12]
    related_event_ids = episide.get("related_events", [])
    related_occasion_ids = episide.get("related_occasions", [])
    related_events = [neo4j_utils.get_entity_by_id(ent_id).name for ent_id in related_event_ids]
    related_occasions = [neo4j_utils.get_entity_by_id(ent_id).name for ent_id in related_occasion_ids]
    related_characters = []
    related_locations = []
    related_times = []
    for ent_id in related_event_ids:
        context = get_event_context(ent_id)
        related_characters.extend(context["characters"])
        related_locations.extend(context["locations"])
        related_times.extend(context["times"])

    related_characters = list(set(related_characters))
    related_locations = list(set(related_locations))
    related_times = list(set(related_times))

    properties = dict()
    if related_events:
        properties["related_events"] = related_events
    if related_occasions:
        properties["related_occasions"] = related_occasions
    if related_characters:
        properties["related_characters"] = related_characters
    if related_locations:
        properties["related_locations"] = related_locations
    if related_times:
        properties["related_times"] = related_times

    existing_episodes_info.append({
        "id": eid,
        "name": name,
        "description": description,
        "source_documents": source_documents,
        "properties": properties,
        "scope": "global"   
    })


In [13]:
existing_episodes_info[0]

{'id': '27754f51edad',
 'name': 'Benny shares his recovery journey',
 'description': 'Benny recounts his long struggle with alcoholism, describing how he secretly continued drinking by burying a bottle in his yard and using a straw to consume it while appearing sober to his wife. He reflects on the absurdity of his behavior and concludes by affirming his eventual path to sobriety through AA, delivering a message of hope and daily commitment to staying sober.',
 'source_documents': ['scene_1_part_1', 'scene_1_part_2'],
 'properties': {'related_events': ['Benny shares his recovery story at the AA meeting (scene_1)'],
  'related_occasions': ['Alcoholics Anonymous Meeting At Church Of St. Peter Los Angeles (scene_1)'],
  'related_characters': ['Benny']},
 'scope': 'global'}

In [14]:
existing_episodes_info[1]

{'id': 'f54145166886',
 'name': 'Chair opens floor for additional shares',
 'description': "Following Benny's testimony, the meeting chair thanks him and announces that there are a few remaining minutes, inviting any other attendees who wish to speak to share their experiences, thereby facilitating the continuation of the meeting's structured format.",
 'source_documents': ['scene_1_part_1', 'scene_1_part_2'],
 'properties': {'related_events': ['Chair invites others to share before closing the meeting (scene_1)'],
  'related_occasions': ['Alcoholics Anonymous Meeting At Church Of St. Peter Los Angeles (scene_1)'],
  'related_characters': ['Chair (scene_1)']},
 'scope': 'global'}

In [15]:
def get_episode_info(episode):
    info = {
        "id": episode.get("id", ""),
        "name": episode.get("name", ""),
        "description": episode.get("description", ""),
        "properties": episode.get("properties", {}),
    }
    return info

In [16]:
subject_info = get_episode_info(episode=existing_episodes_info[0])
object_info = get_episode_info(episode=existing_episodes_info[1])

In [17]:
print(object_info)

{'id': 'f54145166886', 'name': 'Chair opens floor for additional shares', 'description': "Following Benny's testimony, the meeting chair thanks him and announces that there are a few remaining minutes, inviting any other attendees who wish to speak to share their experiences, thereby facilitating the continuation of the meeting's structured format.", 'properties': {'related_events': ['Chair invites others to share before closing the meeting (scene_1)'], 'related_occasions': ['Alcoholics Anonymous Meeting At Church Of St. Peter Los Angeles (scene_1)'], 'related_characters': ['Chair (scene_1)']}}


In [19]:
result = narrative_manager.extract_narrative_relation(subject_entity_info=subject_info, object_entity_info=object_info, entity_type="Episode")

In [20]:
json.loads(result)

{'subject_id': '27754f51edad',
 'object_id': 'f54145166886',
 'relation_type': 'precedes',
 'confidence': 0.95,
 'description': "Benny shares his recovery journey first, and immediately afterward, the chair opens the floor for additional shares, indicating a clear sequential order in the meeting's structure as supported by the related events and shared occasion of the AA meeting."}

In [23]:
{
  "episodes": [
    {
      "name": "Initiate outfit selection and reject initial options",
      "description": "The girls begin selecting outfits for the festivity, with Margie presenting a blue silk dress to Helena and later modeling a black sequined dress. Both options are rejected, reflecting the exploratory and evaluative phase of their dressing ritual, grounded in the shared occasion of preparation.",
      "related_events": ["ent_87f8591d0127", "ent_7101f7ecd815"],
      "related_occasions": ["ent_550932255cc8"]
    },
    {
      "name": "Discuss John's divorce and emotional aftermath",
      "description": "Helena's inquiry about John's divorce prompts Margie to recount the timeline and emotional impact of his separation from Suzanne, a semi-professional model. The conversation reveals John's obsession, downward spiral, and ongoing struggles, forming a reflective narrative unit within the dressing occasion.",
      "related_events": ["ent_a003c52971dd", "ent_afba75b087df"],
      "related_occasions": ["ent_550932255cc8"]
    },
    {
      "name": "Test and confirm suitability of red high heeled shoes",
      "description": "Margie retrieves red satin high heeled shoes and encourages Helena to try them despite her doubts about wearing heels. Helena successfully wears them, marking a turning point in the outfit selection process and building confidence for the next steps.",
      "related_events": ["ent_2be4e3f9db5c", "ent_54dfb64ebdd2", "ent_550932255cc8"]
    },
    {
      "name": "Propose and affirm the sultry red dress for Helena",
      "description": "Following the successful shoe trial, Margie offers to retrieve a sultry red dress she once wore, suggesting it as the ideal match. Helena asks if John would like it, and Margie confidently asserts he would love it, culminating the outfit decision with romantic implication.",
      "related_events": ["ent_bc320ef62e85", "ent_8d9d0c1f31b1", "ent_de462ab5e0e1"],
      "related_occasions": ["ent_550932255cc8"]
    }
  ]
}

{'episodes': [{'name': 'Initiate outfit selection and reject initial options',
   'description': 'The girls begin selecting outfits for the festivity, with Margie presenting a blue silk dress to Helena and later modeling a black sequined dress. Both options are rejected, reflecting the exploratory and evaluative phase of their dressing ritual, grounded in the shared occasion of preparation.',
   'related_events': ['ent_87f8591d0127', 'ent_7101f7ecd815'],
   'related_occasions': ['ent_550932255cc8']},
  {'name': "Discuss John's divorce and emotional aftermath",
   'description': "Helena's inquiry about John's divorce prompts Margie to recount the timeline and emotional impact of his separation from Suzanne, a semi-professional model. The conversation reveals John's obsession, downward spiral, and ongoing struggles, forming a reflective narrative unit within the dressing occasion.",
   'related_events': ['ent_a003c52971dd', 'ent_afba75b087df'],
   'related_occasions': ['ent_550932255cc8'

In [30]:
import json
import networkx as nx
from typing import Dict, Any, List, Tuple, Optional


TYPE_WEIGHT = {
    "causes": 1.0,
    "precedes": 0.7,
    "elaborates": 0.3,
}

SKIP_TYPES = {"conflicts_with"}  # also skip NONE-like if you have them


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _norm_rel_type(x: Any) -> str:
    return str(x or "").strip().lower()


def build_episode_causal_graph(
    episodes_path: str,
    relations_path: str,
    *,
    min_confidence: float = 0.0,
) -> nx.DiGraph:
    """
    Build Episode-only causal graph:
      - skip conflicts_with
      - map all other supported relation types into predicate='causal_link'
      - edge weight = confidence * type_weight
      - elaborates direction is flipped: (A elaborates B) => B -> A with type_weight=0.3
    """
    episodes = load_json(episodes_path)
    relations = load_json(relations_path)

    G = nx.DiGraph()

    # nodes
    for ep in episodes:
        if not isinstance(ep, dict):
            continue
        ep_id = ep.get("id")
        if not ep_id:
            continue
        G.add_node(
            ep_id,
            name=ep.get("name"),
            description=ep.get("description"),
            properties=ep.get("properties", {}),
            embedding=ep.get("embedding"),
            source_documents=ep.get("source_documents", []),
        )

    # edges
    for r in relations:
        if not isinstance(r, dict):
            continue

        s = r.get("subject_id")
        o = r.get("object_id")
        if not s or not o:
            continue

        rel_type = _norm_rel_type(r.get("predicate") or r.get("relation_type"))
        if not rel_type or rel_type in SKIP_TYPES:
            continue
        if rel_type not in TYPE_WEIGHT:
            continue

        conf = float(r.get("confidence", 1.0))
        if conf < float(min_confidence):
            continue

        # elaborates: flip direction
        if rel_type == "elaborates":
            s, o = o, s

        if s not in G or o not in G:
            continue

        type_w = TYPE_WEIGHT[rel_type]
        eff_w = conf * type_w

        # keep original info in attrs
        G.add_edge(
            s,
            o,
            predicate="causal_link",
            original_type=rel_type,
            confidence=conf,
            type_weight=type_w,
            effective_weight=eff_w,
            description=r.get("description"),
            properties=r.get("properties", {}),
            source_documents=r.get("source_documents", []),
        )

    return G


def find_triangle_redundancy_candidates(
    G: nx.DiGraph,
) -> List[Dict[str, Any]]:
    """
    Identify triangles where A->B, B->C, and A->C all exist (directed).

    Returns list of dicts with:
      - a,b,c
      - edge attrs for (a,b), (b,c), (a,c)
    """
    out: List[Dict[str, Any]] = []

    # For each A, iterate B in succ(A), then C in succ(B), check if C in succ(A)
    for a in G.nodes:
        succ_a = set(G.successors(a))
        if not succ_a:
            continue
        for b in succ_a:
            succ_b = set(G.successors(b))
            if not succ_b:
                continue
            for c in succ_b:
                if c == a or c == b:
                    continue
                if c in succ_a:
                    out.append(
                        {
                            "a": a,
                            "b": b,
                            "c": c,
                            "ab": dict(G[a][b]),
                            "bc": dict(G[b][c]),
                            "ac": dict(G[a][c]),
                        }
                    )
    return out



In [42]:
episodes_path = "data/narrative_graph/global/episodes.json"
relations_path = "data/narrative_graph/global/episode_relations.json"

G = build_episode_causal_graph(
    episodes_path=episodes_path,
    relations_path=relations_path,
    min_confidence=0.5,
)

triangles = find_triangle_redundancy_candidates(G)
print("nodes:", G.number_of_nodes())
print("edges:", G.number_of_edges())
print("triangles:", len(triangles))


nodes: 151
edges: 368
triangles: 314


In [43]:
import networkx as nx
from typing import Any, Dict, List


def find_strongly_connected_components(
    G: nx.DiGraph,
    *,
    min_size: int = 2,
    sort_by_size_desc: bool = True,
    include_subgraph: bool = False,
) -> List[Dict[str, Any]]:
    """
    Identify SCCs (strongly connected components) in a directed graph.

    Returns a list of dicts:
      - size
      - nodes (list)
      - edge_count (internal edges)
      - has_cycle (True if size>1 or self-loop)
      - subgraph (optional, nx.DiGraph)
    """
    comps = list(nx.strongly_connected_components(G))

    out: List[Dict[str, Any]] = []
    for cset in comps:
        if not cset:
            continue

        nodes = list(cset)
        size = len(nodes)

        # size==1 can still be a cycle if self-loop exists
        has_self_loop = False
        if size == 1:
            n = nodes[0]
            has_self_loop = G.has_edge(n, n)

        if size < int(min_size) and not has_self_loop:
            continue

        sg = G.subgraph(nodes).copy()
        edge_count = sg.number_of_edges()

        item: Dict[str, Any] = {
            "size": size,
            "nodes": nodes,
            "edge_count": edge_count,
            "has_cycle": True if (size > 1 or has_self_loop) else False,
        }
        if include_subgraph:
            item["subgraph"] = sg

        out.append(item)

    if sort_by_size_desc:
        out.sort(key=lambda x: (int(x.get("size", 0)), int(x.get("edge_count", 0))), reverse=True)

    return out


# Example:
# sccs = find_strongly_connected_components(G, min_size=2)
# print("SCC count:", len(sccs))
# print(sccs[0])


In [44]:
sccs = find_strongly_connected_components(G, min_size=2)

In [45]:
len(sccs)

1